# Computer Vision Overview — Exercises

Companion to the note [Computer Vision Overview](Computer%20Vision%20Overview.md).

Practise the task/metric toolbox of computer vision: IoU, Dice, top-k accuracy, average precision and mAP, non-maximum suppression, segmentation metrics, box-aware augmentation, CutMix and ViT patchification. All implemented in NumPy and checked against sklearn or brute-force pixel counting.

**16 exercises** · 🧠 Concept ×3 · ✍️ By hand ×5 · 💻 Code ×8

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from sklearn.metrics import top_k_accuracy_score, jaccard_score, f1_score
rng = np.random.default_rng(5)

def box_mask(box, H=40, W=40):
    """Rasterise an integer box (x1, y1, x2, y2) into a boolean H x W mask (pixels x1 <= x < x2, y1 <= y < y2)."""
    m = np.zeros((H, W), bool)
    x1, y1, x2, y2 = map(int, box)
    m[y1:y2, x1:x2] = True
    return m

## Part A · Concepts

### Exercise 1 · Match the task
*🧠 Concept · ★☆☆*

For each application name the task from the note's table, its output, and a suitable model:

(a) counting cars in traffic camera frames; (b) measuring tumour area in an MRI slice; (c) separating overlapping people in a crowd photo;
(d) searching a photo library with the text "dog on a beach"; (e) tracking a tennis player's elbow angle.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Distinguish boxes, per-pixel classes, per-object masks, keypoints and image–text embeddings.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) **Object detection** (boxes + labels) with YOLO or Faster R-CNN; count the boxes. (b) **Semantic segmentation** (per-pixel class) with U-Net; area = pixel count × pixel size.
(c) **Instance segmentation** (per-object masks) with Mask R-CNN or SAM, because semantic segmentation would merge touching people.
(d) **Vision–language retrieval** with CLIP: embed the text and all images, rank by cosine similarity. (e) **Keypoints/pose** with HRNet or OpenPose.

</details>

### Exercise 2 · One-stage, two-stage, and DETR
*🧠 Concept · ★★☆*

1. What are the two stages of Faster R-CNN, and why are one-stage detectors (YOLO, SSD) faster?
2. Why do most detectors need **non-maximum suppression** (NMS), and why does DETR not?
3. Why is mAP@[.5:.95] (COCO) a stricter metric than mAP@0.5 (Pascal VOC)?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think about how many candidate boxes each method produces per object.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Stage 1: a Region Proposal Network proposes ~2000 class-agnostic boxes; stage 2: RoI pooling + a head classifies and refines each proposal.
   One-stage detectors predict class scores and box offsets densely on a grid of anchors in a single pass, with no per-proposal second network.
2. Dense predictors fire many overlapping boxes for the same object; NMS keeps the highest-scoring one and removes boxes with IoU above a threshold (Exercise 10).
   DETR predicts a fixed set of $N$ boxes and is trained with **Hungarian (bipartite) matching** to the ground truth, so duplicates are penalised during training and no NMS is needed.
3. It averages AP over IoU thresholds $0.5, 0.55, \dots, 0.95$, so it rewards precise localisation; a box with IoU 0.6 counts as correct only at 3 of 10 thresholds (Exercise 8).

</details>

### Exercise 3 · Augmentation and transfer learning per task
*🧠 Concept · ★★☆*

The note lists flip, crop, colour jitter, mixup/cutmix and transfer learning. For (a) classification, (b) detection, (c) semantic segmentation:
what must happen to the **labels** when you flip or crop the image? Which augmentation should not be applied to the label, and which are unsafe for some datasets?
When would you freeze most of a pretrained backbone?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Geometric transforms change where things are; photometric transforms do not.

</details>

<details>
<summary><b>✅ Solution</b></summary>

- Geometric (flip, crop, rotate): (a) labels unchanged; (b) boxes must be transformed identically, and boxes cut by a crop clipped or dropped; (c) the mask gets the **same**
  transform, with nearest-neighbour interpolation (never bilinear on class ids).
- Photometric (colour jitter, blur, noise): image only, never the mask.
- Unsafe: horizontal flips for text, digits, or left/right-specific labels (e.g. "left hand"); vertical flips for street scenes; strong colour jitter if colour is the label (ripe vs unripe fruit).
- Mixup/CutMix: mix labels in proportion (Exercise 15).
- Freeze most of the backbone when the target dataset is small and similar to the pretraining data; fine-tune more layers (with a lower LR) when you have more data or a domain shift (medical, satellite) ([[Training Tricks]]).

</details>

## Part B · By hand

### Exercise 4 · IoU by hand
*✍️ By hand · ★☆☆*

Boxes in $(x_1, y_1, x_2, y_2)$ format: $A = (0,0,4,4)$, $B = (2,2,6,6)$. Compute the intersection, union and IoU.

In [ ]:
inter_ab = None
union_ab = None
iou_ab = None

_ma, _mb = box_mask((0, 0, 4, 4)), box_mask((2, 2, 6, 6))
check('intersection', inter_ab, (_ma & _mb).sum()); check('union', union_ab, (_ma | _mb).sum())
check('IoU', iou_ab, (_ma & _mb).sum() / (_ma | _mb).sum(), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Intersection box: $(\max x_1, \max y_1, \min x_2, \min y_2)$. Union $= |A|+|B|-|A\cap B|$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Intersection $(2,2,4,4)$, area 4. Union $16+16-4 = 28$. IoU $= 4/28 = 1/7 \approx 0.143$, far below the usual 0.5 match threshold.

```python
inter_ab, union_ab, iou_ab = 4, 28, 1 / 7
```

</details>

### Exercise 5 · Dice vs IoU
*✍️ By hand · ★☆☆*

For sets (masks) $A, B$: $\mathrm{IoU} = \frac{|A\cap B|}{|A\cup B|}$ and $\mathrm{Dice} = \frac{2|A\cap B|}{|A|+|B|}$.
Show that $\mathrm{Dice} = \frac{2J}{1+J}$ with $J$ the IoU, compute Dice for Exercise 4, and say which is always larger.

In [ ]:
dice_ab = None

_ma, _mb = box_mask((0, 0, 4, 4)), box_mask((2, 2, 6, 6))
check('Dice', dice_ab, 2 * (_ma & _mb).sum() / (_ma.sum() + _mb.sum()), tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Use $|A|+|B| = |A\cup B| + |A\cap B|$ and divide numerator and denominator by $|A\cup B|$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\mathrm{Dice} = \frac{2I}{U + I} = \frac{2I/U}{1 + I/U} = \frac{2J}{1+J}$. For $J = 1/7$: $\frac{2/7}{8/7} = 0.25$.
Since $1+J \le 2$, Dice $\ge$ IoU (equal only at 0 and 1). Dice equals the F1 score of the pixel classification (Exercise 12), which is why it is popular in medical segmentation.

```python
dice_ab = 0.25
```

</details>

### Exercise 6 · Average precision by hand
*✍️ By hand · ★★☆*

A detector returns 5 detections for one class, sorted by confidence, matched to the 4 ground-truth objects as: **TP, FP, TP, TP, FP**.
Compute precision and recall after each detection, then the all-point interpolated AP
$\mathrm{AP} = \sum_n (R_n - R_{n-1})\,P^{\text{interp}}(R_n)$ with $P^{\text{interp}}(r) = \max_{r'\ge r} P(r')$.

In [ ]:
ap_hand = None

_tp = np.array([1, 0, 1, 1, 0]); _P = np.cumsum(_tp) / np.arange(1, 6); _R = np.cumsum(_tp) / 4
_ap = sum((_R[n] - (_R[n - 1] if n else 0)) * _P[n:].max() for n in range(5))   # P[n:].max() is the interpolation since R is non-decreasing
check('AP', ap_hand, _ap, tol=1e-4)

<details>
<summary><b>💡 Hint 1</b></summary>

Precision: $1, 1/2, 2/3, 3/4, 3/5$. Recall: $1/4, 1/4, 2/4, 3/4, 3/4$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Recall increases at detections 1, 3, 4 (by $0.25$ each). Interpolated precision there: $\max(1, .5, .667, .75, .6)=1$, then $0.75$, $0.75$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

| # | TP? | P | R |
|---|---|---|---|
| 1 | ✔ | 1.000 | 0.25 |
| 2 | ✘ | 0.500 | 0.25 |
| 3 | ✔ | 0.667 | 0.50 |
| 4 | ✔ | 0.750 | 0.75 |
| 5 | ✘ | 0.600 | 0.75 |

$\mathrm{AP} = 0.25\cdot1 + 0.25\cdot0.75 + 0.25\cdot0.75 = 0.625$. The missed 4th object caps recall at $0.75$, so AP can be at most $0.75$ here.
mAP is the mean of AP over classes (and, for COCO, over IoU thresholds).

```python
ap_hand = 0.625
```

</details>

### Exercise 7 · ViT tokens and patch embedding
*✍️ By hand · ★☆☆*

ViT-Base splits a $224\times224\times3$ image into $16\times16$ patches and linearly embeds each into $d = 768$ dimensions, then prepends a [CLS] token.
(a) How many tokens enter the Transformer? (b) How many parameters (with bias) does the patch-embedding layer have?
(c) By what factor does the self-attention cost grow at $448\times448$ with the same patch size?

In [ ]:
n_tokens = None
patch_params = None
attn_factor = None

check('tokens', n_tokens, (224 // 16) ** 2 + 1)
check('patch embedding params', patch_params, 16 * 16 * 3 * 768 + 768)
check('attention cost factor', attn_factor, (((448 // 16) ** 2 + 1) / ((224 // 16) ** 2 + 1)) ** 2, tol=0.05)

<details>
<summary><b>💡 Hint</b></summary>

$(224/16)^2 = 14^2$. The embedding is a linear map from $16\cdot16\cdot3$ to 768 (equivalently a $16\times16$ conv with stride 16).

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $14^2 + 1 = 197$. (b) $768\cdot768 + 768 = 590{,}592$ (the flattened patch has $16\cdot16\cdot3 = 768$ values).
(c) Tokens grow to $28^2+1 = 785$; attention is $O(n^2)$, so the factor is $(785/197)^2 \approx 15.9$ (≈ 16). See [[Transformers]].

```python
n_tokens = 197
patch_params = 590592
attn_factor = (785 / 197) ** 2
```

</details>

### Exercise 8 · IoU thresholds in COCO mAP
*✍️ By hand · ★☆☆*

COCO evaluates at IoU thresholds $\{0.50, 0.55, \dots, 0.95\}$ (10 values). A detection overlaps its object with IoU $0.72$.
At what fraction of the thresholds does it count as a true positive? What about IoU $0.96$ and $0.49$?

In [ ]:
frac_072 = None
frac_096 = None
frac_049 = None

_th = np.linspace(0.5, 0.95, 10)
check('IoU 0.72', frac_072, np.mean(0.72 >= _th)); check('IoU 0.96', frac_096, np.mean(0.96 >= _th)); check('IoU 0.49', frac_049, np.mean(0.49 >= _th))

<details>
<summary><b>💡 Hint</b></summary>

List the thresholds and count those $\le$ the IoU.

</details>

<details>
<summary><b>✅ Solution</b></summary>

IoU 0.72 passes $0.50, 0.55, 0.60, 0.65, 0.70$: $5/10 = 0.5$. IoU 0.96 passes all ($1.0$); 0.49 none ($0$). Tight boxes matter for COCO mAP.

```python
frac_072, frac_096, frac_049 = 0.5, 1.0, 0.0
```

</details>

## Part C · Code from scratch

### Exercise 9 · Vectorised pairwise IoU
*💻 Code · ★☆☆*

Implement `box_iou(A, B)` for arrays of boxes `A` (n×4) and `B` (m×4) in $(x_1,y_1,x_2,y_2)$ format, returning the $n\times m$ IoU matrix without Python loops.

In [ ]:
def box_iou(A, B):
    return None

def rand_boxes(n):
    xy = rng.integers(0, 30, size=(n, 2)); wh = rng.integers(1, 10, size=(n, 2))
    return np.c_[xy, xy + wh].astype(float)

A_b, B_b = rand_boxes(6), rand_boxes(5)
iou_mat = box_iou(A_b, B_b)

_ref = np.array([[(box_mask(a) & box_mask(b)).sum() / (box_mask(a) | box_mask(b)).sum() for b in B_b] for a in A_b])
check('IoU matrix = pixel counting', iou_mat, _ref)

<details>
<summary><b>💡 Hint</b></summary>

Broadcast: `x1 = np.maximum(A[:, None, 0], B[None, :, 0])`, etc.; clip widths/heights at 0 with `np.clip(x2 - x1, 0, None)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Broadcasting turns the double loop into array operations; this is exactly `torchvision.ops.box_iou`.

```python
def box_iou(A, B):
    area = lambda X: (X[:, 2] - X[:, 0]) * (X[:, 3] - X[:, 1])
    lt = np.maximum(A[:, None, :2], B[None, :, :2])
    rb = np.minimum(A[:, None, 2:], B[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    return inter / (area(A)[:, None] + area(B)[None, :] - inter)

def rand_boxes(n):
    xy = rng.integers(0, 30, size=(n, 2)); wh = rng.integers(1, 10, size=(n, 2))
    return np.c_[xy, xy + wh].astype(float)

A_b, B_b = rand_boxes(6), rand_boxes(5)
iou_mat = box_iou(A_b, B_b)
```

</details>

### Exercise 10 · Non-maximum suppression
*💻 Code · ★★☆*

Implement greedy `nms(boxes, scores, thr)`: repeatedly keep the highest-scoring remaining box and discard all remaining boxes whose IoU with it exceeds `thr`.
Return the kept indices in order of decreasing score.

In [ ]:
def nms(boxes, scores, thr=0.5):
    return None

toy_boxes = np.array([[0, 0, 10, 10], [1, 1, 11, 11], [20, 20, 30, 30], [0, 0, 9, 10], [21, 19, 31, 29]], float)
toy_scores = np.array([0.9, 0.8, 0.7, 0.95, 0.6])
kept_toy = nms(toy_boxes, toy_scores, 0.5)
many_boxes = rand_boxes(60); many_scores = rng.random(60)
kept_many = nms(many_boxes, many_scores, 0.3)

check('toy example', kept_toy, [3, 2])
if kept_many is not None:
    _I = box_iou(many_boxes, many_boxes); _k = list(kept_many)
    _ok1 = all(_I[i, j] <= 0.3 for a, i in enumerate(_k) for j in _k[a + 1:])
    _ok2 = all(any(_I[r, k] > 0.3 and many_scores[k] >= many_scores[r] for k in _k) for r in set(range(60)) - set(_k))
    check('kept boxes do not overlap > thr', _ok1, True); check('every removed box is covered by a better kept box', _ok2, True)
else:
    check('NMS properties', None, True)

<details>
<summary><b>💡 Hint</b></summary>

`order = np.argsort(-scores)`; while `order` is non-empty: `i = order[0]`, keep it, `order = order[1:][iou(i, order[1:]) <= thr]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

In the toy example box 3 (0.95) suppresses boxes 0 (IoU 0.9) and 1 (IoU ≈ 0.61); box 2 (0.7) suppresses box 4 (IoU ≈ 0.68), so `[3, 2]` remain.
NMS is class-wise in practice; Soft-NMS decays scores instead of deleting, which helps with crowded scenes where true objects overlap.

```python
def nms(boxes, scores, thr=0.5):
    order = np.argsort(-scores)
    keep = []
    while len(order):
        i = order[0]; keep.append(int(i))
        rest = order[1:]
        order = rest[box_iou(boxes[i:i + 1], boxes[rest])[0] <= thr]
    return keep

toy_boxes = np.array([[0, 0, 10, 10], [1, 1, 11, 11], [20, 20, 30, 30], [0, 0, 9, 10], [21, 19, 31, 29]], float)
toy_scores = np.array([0.9, 0.8, 0.7, 0.95, 0.6])
kept_toy = nms(toy_boxes, toy_scores, 0.5)
many_boxes = rand_boxes(60); many_scores = rng.random(60)
kept_many = nms(many_boxes, many_scores, 0.3)
```

</details>

### Exercise 11 · Top-1 and top-5 accuracy
*💻 Code · ★☆☆*

Implement `top_k_acc(scores, y, k)` (scores of shape `(n, C)`) and evaluate top-1 and top-5 on random scores for 1000 classes, then on scores that are slightly informative.

In [ ]:
def top_k_acc(scores, y, k):
    return None

n_s, C_s = 2000, 1000
y_s = rng.integers(0, C_s, n_s)
scores_s = rng.normal(size=(n_s, C_s)); scores_s[np.arange(n_s), y_s] += 2.5   # a weak signal
top1, top5 = top_k_acc(scores_s, y_s, 1), top_k_acc(scores_s, y_s, 5)

check('top-1', top1, top_k_accuracy_score(y_s, scores_s, k=1, labels=np.arange(C_s)))
check('top-5', top5, top_k_accuracy_score(y_s, scores_s, k=5, labels=np.arange(C_s)))

<details>
<summary><b>💡 Hint</b></summary>

`np.argpartition(-scores, k-1, axis=1)[:, :k]` gives the top-k class indices (unsorted); then test membership of `y`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Chance levels are $1/1000$ and $5/1000$. Top-5 was introduced for ImageNet because many images contain several objects or fine-grained classes
(e.g. 120 dog breeds), so a single label is often ambiguous.

```python
def top_k_acc(scores, y, k):
    topk = np.argpartition(-scores, k - 1, axis=1)[:, :k]
    return np.mean(np.any(topk == y[:, None], axis=1))

n_s, C_s = 2000, 1000
y_s = rng.integers(0, C_s, n_s)
scores_s = rng.normal(size=(n_s, C_s)); scores_s[np.arange(n_s), y_s] += 2.5
top1, top5 = top_k_acc(scores_s, y_s, 1), top_k_acc(scores_s, y_s, 5)
print(top1, top5)
```

</details>

### Exercise 12 · Segmentation metrics from a confusion matrix
*💻 Code · ★★☆*

Given integer label maps `gt` and `pred` (values $0..C-1$), implement `seg_metrics(gt, pred, C)` returning `(iou_per_class, miou, dice_per_class)`
via the $C\times C$ confusion matrix (`np.bincount` trick). Then explain why pixel accuracy is misleading on the example (90% background).

In [ ]:
def seg_metrics(gt, pred, C):
    return None, None, None

gt_s = np.zeros((64, 64), int); gt_s[10:20, 10:30] = 1; gt_s[40:50, 40:46] = 2
pred_s = np.zeros_like(gt_s); pred_s[12:22, 10:30] = 1; pred_s[40:50, 42:46] = 2; pred_s[0:3, 0:3] = 2
iou_c, miou, dice_c = seg_metrics(gt_s, pred_s, 3)
pixel_acc = np.mean(gt_s == pred_s)

check('IoU per class', iou_c, jaccard_score(gt_s.ravel(), pred_s.ravel(), average=None))
check('mIoU', miou, jaccard_score(gt_s.ravel(), pred_s.ravel(), average='macro'))
check('Dice per class = F1', dice_c, f1_score(gt_s.ravel(), pred_s.ravel(), average=None))

<details>
<summary><b>💡 Hint 1</b></summary>

`cm = np.bincount(C * gt.ravel() + pred.ravel(), minlength=C*C).reshape(C, C)` (rows = truth).

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

$TP = \mathrm{diag}(cm)$, $FP$ = column sums − TP, $FN$ = row sums − TP.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$IoU_c = \frac{TP}{TP+FP+FN}$, $Dice_c = \frac{2TP}{2TP+FP+FN}$. Pixel accuracy is ≈ 0.97 here although class 2 has IoU only ≈ 0.58 (class 1: 0.67):
background dominates the pixel count. mIoU weights classes equally, which is why it is the standard segmentation metric.

```python
def seg_metrics(gt, pred, C):
    cm = np.bincount(C * gt.ravel() + pred.ravel(), minlength=C * C).reshape(C, C)
    tp = np.diag(cm).astype(float); fp = cm.sum(0) - tp; fn = cm.sum(1) - tp
    iou = tp / (tp + fp + fn)
    return iou, iou.mean(), 2 * tp / (2 * tp + fp + fn)

gt_s = np.zeros((64, 64), int); gt_s[10:20, 10:30] = 1; gt_s[40:50, 40:46] = 2
pred_s = np.zeros_like(gt_s); pred_s[12:22, 10:30] = 1; pred_s[40:50, 42:46] = 2; pred_s[0:3, 0:3] = 2
iou_c, miou, dice_c = seg_metrics(gt_s, pred_s, 3)
pixel_acc = np.mean(gt_s == pred_s)
print(iou_c, miou, pixel_acc)
```

</details>

### Exercise 13 · Average precision for detection
*💻 Code · ★★☆*

Implement `average_precision(tp, scores, n_gt)` (all-point interpolation): sort detections by score, accumulate TP/FP, compute precision/recall,
make precision monotonically non-increasing from the right, and sum $\Delta R\cdot P^{\text{interp}}$. Reproduce Exercise 6, then test a perfect ranking.

In [ ]:
def average_precision(tp, scores, n_gt):
    return None

ap_toy = average_precision(np.array([1, 0, 1, 1, 0]), np.array([.9, .8, .7, .6, .5]), 4)
ap_shuffled = average_precision(np.array([0, 1, 1, 1, 0]), np.array([.8, .9, .6, .7, .5]), 4)   # same detections, unsorted input
ap_perfect = average_precision(np.array([1, 1, 1, 0, 0]), np.array([.9, .8, .7, .2, .1]), 3)

check('Exercise 6 value', ap_toy, 0.625)
check('order of the input does not matter', ap_shuffled, 0.625)
check('perfect ranking -> AP = 1', ap_perfect, 1.0)

<details>
<summary><b>💡 Hint</b></summary>

`P_interp = np.maximum.accumulate(P[::-1])[::-1]`; `dR = np.diff(np.r_[0, R])`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The right-to-left running maximum implements $\max_{r'\ge r}P(r')$, which removes the zig-zag of the PR curve. Pascal VOC (2010+) uses exactly this;
COCO samples the interpolated curve at 101 recall points and averages over IoU thresholds and classes.

```python
def average_precision(tp, scores, n_gt):
    order = np.argsort(-scores)
    tp = np.asarray(tp)[order].astype(float)
    ctp, cfp = np.cumsum(tp), np.cumsum(1 - tp)
    P, R = ctp / (ctp + cfp), ctp / n_gt
    P_interp = np.maximum.accumulate(P[::-1])[::-1]
    return float(np.sum(np.diff(np.r_[0, R]) * P_interp))

ap_toy = average_precision(np.array([1, 0, 1, 1, 0]), np.array([.9, .8, .7, .6, .5]), 4)
ap_shuffled = average_precision(np.array([0, 1, 1, 1, 0]), np.array([.8, .9, .6, .7, .5]), 4)
ap_perfect = average_precision(np.array([1, 1, 1, 0, 0]), np.array([.9, .8, .7, .2, .1]), 3)
```

</details>

### Exercise 14 · Flipping images together with their boxes
*💻 Code · ★★☆*

Implement `hflip_with_boxes(img, boxes)` for an image of width $W$ and boxes $(x_1,y_1,x_2,y_2)$ in pixel-edge coordinates.
Verify that the rasterised flipped box coincides with the flipped rasterised box.

In [ ]:
def hflip_with_boxes(img, boxes):
    return None, None

img_f = rng.random((40, 40)); boxes_f = rand_boxes(4)
img_ff, boxes_ff = hflip_with_boxes(img_f, boxes_f)

check('image flipped', img_ff, img_f[:, ::-1])
check('box masks consistent', None if boxes_ff is None else np.array([box_mask(b) for b in boxes_ff]), np.array([box_mask(b)[:, ::-1] for b in boxes_f]))

<details>
<summary><b>💡 Hint</b></summary>

A pixel edge at $x$ maps to $W - x$; so $x_1' = W - x_2$ and $x_2' = W - x_1$ (swap to keep $x_1' < x_2'$).

</details>

<details>
<summary><b>✅ Solution</b></summary>

$x_1' = W - x_2$, $x_2' = W - x_1$, $y$ unchanged. A classic bug is $x' = W - 1 - x$, which is correct for pixel *centres/indices* but off by one for edge coordinates.
Libraries like Albumentations handle these conventions for you; the check by rasterisation is a good habit.

```python
def hflip_with_boxes(img, boxes):
    W = img.shape[1]
    b = boxes.copy()
    b[:, 0], b[:, 2] = W - boxes[:, 2], W - boxes[:, 0]
    return img[:, ::-1], b

img_f = rng.random((40, 40)); boxes_f = rand_boxes(4)
img_ff, boxes_ff = hflip_with_boxes(img_f, boxes_f)
```

</details>

### Exercise 15 · CutMix
*💻 Code · ★★☆*

CutMix pastes a rectangle from image $x_b$ into image $x_a$ and mixes the one-hot labels with $\lambda = 1 - \frac{\text{area of the patch}}{HW}$:
$\tilde y = \lambda y_a + (1-\lambda) y_b$. Implement `cutmix(xa, xb, ya, yb, box)` with `box = (x1, y1, x2, y2)` returning `(x_mix, y_mix)`.
Verify with a black and a white image that the label weight equals the actual pixel fraction. How does it differ from mixup?

In [ ]:
def cutmix(xa, xb, ya, yb, box):
    return None, None

xa_c, xb_c = np.zeros((32, 32)), np.ones((32, 32))
ya_c, yb_c = np.eye(10)[3], np.eye(10)[7]
x_mix, y_mix = cutmix(xa_c, xb_c, ya_c, yb_c, (4, 8, 20, 18))

check('pixels pasted', None if x_mix is None else x_mix.sum(), box_mask((4, 8, 20, 18), 32, 32).sum())
check('label weight of class 7 = pixel fraction from xb', None if y_mix is None else y_mix[7], None if x_mix is None else x_mix.mean())
check('labels sum to 1', None if y_mix is None else y_mix.sum(), 1.0)

<details>
<summary><b>💡 Hint</b></summary>

`x = xa.copy(); x[y1:y2, x1:x2] = xb[y1:y2, x1:x2]`; patch area $(x_2-x_1)(y_2-y_1)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Here the patch is $16\times10 = 160$ px of $1024$, so $\lambda = 0.84375$ and class 7 gets weight $0.15625$.
**Mixup** blends whole images, $\tilde x = \lambda x_a + (1-\lambda)x_b$, producing unnatural ghosted images; CutMix keeps every pixel natural and
forces the model to recognise objects from partial views (regularisation similar to cutout, but no pixels are wasted).

```python
def cutmix(xa, xb, ya, yb, box):
    x1, y1, x2, y2 = box
    x = xa.copy()
    x[y1:y2, x1:x2] = xb[y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / xa.size
    return x, lam * ya + (1 - lam) * yb

xa_c, xb_c = np.zeros((32, 32)), np.ones((32, 32))
ya_c, yb_c = np.eye(10)[3], np.eye(10)[7]
x_mix, y_mix = cutmix(xa_c, xb_c, ya_c, yb_c, (4, 8, 20, 18))
```

</details>

### Exercise 16 · Patchify for a Vision Transformer
*💻 Code · ★★☆*

Implement `patchify(X, P)` turning a batch `X` of shape `(B, C, H, W)` into `(B, N, C·P·P)` with $N = (H/P)(W/P)$ patches in row-major order,
each flattened in $(C, P, P)$ order; and `unpatchify(T, P, C, H, W)` that inverts it. Use only `reshape`/`transpose`.

In [ ]:
def patchify(X, P):
    return None

def unpatchify(T, P, C, H, W):
    return None

X_p = rng.normal(size=(2, 3, 32, 48))
T_p = patchify(X_p, 16)

check('shape', None if T_p is None else T_p.shape, (2, 6, 768))
check('patch (row 1, col 2) of image 1', None if T_p is None else T_p[1, 1 * 3 + 2], X_p[1, :, 16:32, 32:48].ravel())
check('unpatchify inverts patchify', None if T_p is None else unpatchify(T_p, 16, 3, 32, 48), X_p)

<details>
<summary><b>💡 Hint 1</b></summary>

Reshape to `(B, C, H/P, P, W/P, P)`.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Transpose to `(B, H/P, W/P, C, P, P)` and reshape to `(B, N, C*P*P)`. Unpatchify does the reverse.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Patchify plus a linear layer is exactly a $P\times P$ conv with stride $P$ (the ViT "patch embedding"). MAE masks ~75% of these tokens and reconstructs them,
using `unpatchify` to turn predictions back into an image.

```python
def patchify(X, P):
    B, C, H, W = X.shape
    T = X.reshape(B, C, H // P, P, W // P, P).transpose(0, 2, 4, 1, 3, 5)
    return T.reshape(B, (H // P) * (W // P), C * P * P)

def unpatchify(T, P, C, H, W):
    B = T.shape[0]
    X = T.reshape(B, H // P, W // P, C, P, P).transpose(0, 3, 1, 4, 2, 5)
    return X.reshape(B, C, H, W)

X_p = rng.normal(size=(2, 3, 32, 48))
T_p = patchify(X_p, 16)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Computer Vision Overview](Computer%20Vision%20Overview.md).*